# The Poisson Equation and Its Parallel Solution

This notebook treats the **Poisson equation**, the prototype of an elliptic partial differential equation, and its parallel solution with MPI.

We discretize the equation with finite differences, compare the iterative solvers Jacobi, Gauss–Seidel, SOR and red-black SOR, and parallelize them with a 1D domain decomposition.

**After working through this notebook you will be able to**

* discretize the Poisson equation with finite differences, including Dirichlet and Neumann boundary conditions,
* solve the resulting linear system with Jacobi, Gauss–Seidel and (red-black) SOR, and judge convergence with the residual,
* parallelize a stencil solver with a 1D domain decomposition, ghost layers, halo exchange and global reductions,
* recognize when the parallelization changes the algorithm — and how to avoid it.

Prompts marked with 🤔 **Predict** ask you to think about the result *before* running the next cell.

## Working with this Notebook

The C cells are compiled and run with two cell magics: `%%c` for sequential programs and `%%mpi` for MPI programs.
The content of a cell is the body of `main()`; in `%%mpi` cells, MPI is already initialized and finalized, and `rank` and `size` (of `MPI_COMM_WORLD`) are available.
The options used in this notebook are

| Option | Meaning |
|---|---|
| `-n N` | start `N` MPI processes (default: 4) |
| `-a ARG …` | pass command line arguments to the program (`argc`, `argv` are available in the cell) |
| `-i HEADER …` | include additional headers, e.g. `-i math.h` |
| `-f FLAG …` | additional compiler flags without the leading `-`: `-f lm` links the math library |
| `-O VAR` | store the program output in the Python variable `VAR` |
| `-o FILE` | where to write the source code |
| `-r` | treat the cell as a complete program instead of the body of `main()` |

Plots need numpy and matplotlib.

In [ ]:
%load_ext ice.magic
import numpy as np
import matplotlib.pyplot as plt

## The Mathematical Model

A mathematical model describes how physical quantities relate to each other in a natural phenomenon — often as a **partial differential equation** (PDE) with derivatives in space and time.
Analytical solutions exist only for special cases; in general, the equations must be solved numerically.

The **Poisson equation** for a quantity $p$ on a domain $\Omega$ reads

$$\Delta p = \frac{\partial^2 p}{\partial x^2} + \frac{\partial^2 p}{\partial y^2} = f \quad \text{in } \Omega,$$

with the **Laplace operator** $\Delta$ and the right-hand side $f$.
For $f = 0$ it is the **Laplace equation**.
Both are *elliptic* PDEs; they describe equilibrium states — steady heat conduction, electrostatic and gravitational potentials, and the pressure in incompressible flow.

### Boundary Conditions

The equation alone does not determine the solution; it needs conditions on the boundary $\partial\Omega$:

* **Dirichlet** condition: the value is prescribed, $p = g$ on $\partial\Omega$,
* **Neumann** condition: the normal derivative is prescribed, $\frac{\partial p}{\partial n} = h$ on $\partial\Omega$ — *homogeneous* for $h = 0$.

We consider homogeneous Neumann conditions on the whole boundary, as for the pressure in incompressible flow.
This has two consequences:

* the solution is unique only **up to a constant** — with $p$, also $p + c$ is a solution,
* a solution exists only if the right-hand side is **compatible**: integrating $\Delta p = f$ over $\Omega$ and applying the divergence theorem gives $\int_\Omega f \, dx = \int_{\partial\Omega} \frac{\partial p}{\partial n} \, ds = 0$.

Iterative solvers still work — they converge to one of the solutions — but comparisons must ignore the constant, and an incompatible right-hand side (e.g. from rounding) makes the residual stagnate.

## Discretization with Finite Differences

The domain is covered with a grid of points, the equation is evaluated at these points, and the derivatives are approximated by values at neighboring points.
Of the three basic methods — finite differences (FDM), finite volumes (FVM) and finite elements (FEM) — we use the simplest, **finite differences**.

Motivated by the definition of the derivative, the **central difference** approximates the first derivative, and applying it twice with half the step size gives the second derivative:

$$\left(\frac{\partial p}{\partial x}\right)_i \approx \frac{p_{i+1} - p_{i-1}}{2\,\delta x}, \qquad
\left(\frac{\partial^2 p}{\partial x^2}\right)_i \approx \frac{p_{i+1} - 2 p_i + p_{i-1}}{\delta x^2},$$

both second-order accurate: the error decreases with $\delta x^2$.
In 2D, the discrete Laplacian at point $(i, j)$ is

$$(\Delta p)_{i,j} \approx \frac{p_{i+1,j} - 2p_{i,j} + p_{i-1,j}}{\delta x^2} + \frac{p_{i,j+1} - 2p_{i,j} + p_{i,j-1}}{\delta y^2}.$$

It uses the point and its four neighbors — the **five-point stencil**:

![A grid of 6 by 6 inner cells surrounded by a layer of ghost cells. The five-point stencil connects a point with its four neighbors. The Neumann condition copies the first inner value into the ghost cell](img/cfd-stencil.svg)

We use a **cell-centered** grid: the unknowns sit at the centers of `imax × jmax` cells, $x_i = (i - \frac12)\,\delta x$, and the grid is surrounded by one layer of **ghost cells** that carry the boundary conditions.
The homogeneous Neumann condition becomes $p_{0,j} = p_{1,j}$ on the left boundary (and likewise on the others): the one-sided difference $(p_{1,j} - p_{0,j})/\delta x = 0$ approximates the derivative at the wall, which lies exactly halfway between the two points — so it is even second-order accurate there.

### The Linear System

One equation per inner point gives a **linear system** $A\,p = f$ with the coefficient matrix $A$, the solution vector $p$ and the right-hand side $f$:

![The matrix of the 5-point stencil for a 4 by 4 grid: 16 rows, with the diagonal, two neighboring diagonals for i plus minus 1, and two diagonals at distance 4 for j plus minus 1](img/cfd-matrix.svg)

$A$ is sparse — at most five nonzeros per row — but has `imax · jmax` rows, e.g. a million for a 1000 × 1000 grid.
Direct solvers are too expensive in memory and time; **iterative solvers** apply the stencil directly, without ever storing $A$.

## Iterative Solvers

### The Residual Correction

Let $p$ be an approximation of the solution.
At point $(i, j)$ it leaves the **residual**

$$r_{i,j} = f_{i,j} - (\Delta p)_{i,j}.$$

We look for a correction $c_{i,j}$ of the value at this point alone, $p^*_{i,j} = p_{i,j} + c_{i,j}$, that makes the residual vanish.
The neighbors stay unchanged, so in the discrete Laplacian only the center terms change:

$$(\Delta p^*)_{i,j} = \frac{p_{i+1,j} - 2(p_{i,j} + c_{i,j}) + p_{i-1,j}}{\delta x^2} + \frac{p_{i,j+1} - 2(p_{i,j} + c_{i,j}) + p_{i,j-1}}{\delta y^2}
= (\Delta p)_{i,j} - c_{i,j} \left(\frac{2}{\delta x^2} + \frac{2}{\delta y^2}\right).$$

The new residual is therefore

$$r^*_{i,j} = f_{i,j} - (\Delta p^*)_{i,j} = r_{i,j} + c_{i,j} \left(\frac{2}{\delta x^2} + \frac{2}{\delta y^2}\right).$$

Setting $r^*_{i,j} = 0$ and solving for the correction gives

$$c_{i,j} = -\frac{r_{i,j}}{\frac{2}{\delta x^2} + \frac{2}{\delta y^2}}
= -\frac{r_{i,j}}{\frac{2(\delta y^2 + \delta x^2)}{\delta x^2 \, \delta y^2}}
= -\frac{\delta x^2 \, \delta y^2}{2(\delta x^2 + \delta y^2)} \, r_{i,j},$$

so the corrected value is

$$p_{i,j}^* = p_{i,j} - \frac{\delta x^2 \, \delta y^2}{2(\delta x^2 + \delta y^2)} \, r_{i,j}.$$

The factor in front of $r_{i,j}$ is the `factor` in the codes below (multiplied by $\omega$ for SOR).

Sweeping over the grid and applying this correction to every point again and again is a **relaxation** method.
With $A = L + D + U$ (strictly lower, diagonal, strictly upper part), the variants are

| Method | Iteration | Implementation |
|---|---|---|
| **Jacobi** | $D\,p^{k} = -(L + U)\,p^{k-1} + f$ | two arrays: compute only from the old values |
| **Gauss–Seidel** | $(L + D)\,p^{k} = -U\,p^{k-1} + f$ | one array: use the new values as soon as they are available |
| **SOR** | Gauss–Seidel with the correction multiplied by $\omega \in (1, 2)$ | *successive over-relaxation*: extrapolate the correction |

Written out, an SOR step in lexicographic order (left to right, bottom to top) uses the **new** values of the left and lower neighbors and the **old** ones of the right and upper neighbors:

$$p^{k+1}_{i,j} = (1 - \omega)\, p^k_{i,j} + \frac{\omega}{\frac{2}{\delta x^2} + \frac{2}{\delta y^2}}
\left( \frac{p^{k+1}_{i-1,j} + p^{k}_{i+1,j}}{\delta x^2} + \frac{p^{k+1}_{i,j-1} + p^{k}_{i,j+1}}{\delta y^2} - f_{i,j} \right)$$

### When Is the Solution Good Enough?

The iteration stops when a norm of the residual vector drops below a threshold $\varepsilon$ — commonly the L2 norm, divided by the square root of the number of unknowns so that it doesn't depend on the problem size:

$$\lVert r \rVert = \sqrt{\frac{1}{i_{max}\, j_{max}} \sum_{i,j} r_{i,j}^2} < \varepsilon.$$

The codes compute $r_{i,j}$ while updating, i.e. partly from already updated neighbors — a cheap *pseudo-residual*, which is good enough as a stopping criterion.

### A Test with a Known Solution

To check a solver, use a problem with a known solution — a *manufactured solution*.
$p(x, y) = \cos(\pi x) \cos(\pi y)$ on the unit square satisfies the homogeneous Neumann conditions, and $\Delta p = -2\pi^2 p$ gives the right-hand side; its mean is 0, so it is compatible.

The following program solves this problem with the method and $\omega$ given as command line arguments (`jacobi`, `gs`, `sor`, `rb` for red-black SOR, see below) and reports the iterations and the maximum error — after removing the mean, since the solution is unique only up to a constant.

In [ ]:
%%c -i string.h math.h -f lm -a sor 1.9 64 -o code/cfd-poisson.c
// arguments: method (jacobi | gs | sor | rb), omega, n (cells per direction)
const char *method = argc > 1 ? argv[1] : "sor";
const double omega = argc > 2 ? atof(argv[2]) : 1.9;
const int n        = argc > 3 ? atoi(argv[3]) : 64;

const int imax = n, jmax = n;                   // unit square, n x n cells
const double dx = 1.0 / imax, dy = 1.0 / jmax;
const double eps = 1e-6;                        // stop when the RMS residual is below eps
const int itermax = 100000;
const double PI = 3.14159265358979323846;

const size_t bytes = (size_t)(imax + 2) * (jmax + 2) * sizeof(double);
double *p = calloc(1, bytes), *pold = calloc(1, bytes), *rhs = calloc(1, bytes);
#define P(i, j)    p[(j) * (imax + 2) + (i)]
#define POLD(i, j) pold[(j) * (imax + 2) + (i)]
#define RHS(i, j)  rhs[(j) * (imax + 2) + (i)]

// manufactured solution p = cos(pi x) cos(pi y): homogeneous Neumann conditions, f = -2 pi^2 p
for (int j = 1; j <= jmax; ++j)
    for (int i = 1; i <= imax; ++i)
        RHS(i, j) = -2.0 * PI * PI * cos(PI * (i - 0.5) * dx) * cos(PI * (j - 0.5) * dy);

const int jacobi = strcmp(method, "jacobi") == 0, redblack = strcmp(method, "rb") == 0;
const double w = (jacobi || strcmp(method, "gs") == 0) ? 1.0 : omega;
const double dx2 = dx * dx, dy2 = dy * dy, idx2 = 1.0 / dx2, idy2 = 1.0 / dy2;
const double factor = w * 0.5 * dx2 * dy2 / (dx2 + dy2);

int it = 0;
double res = 1.0;
while (res >= eps * eps && it < itermax) {
    res = 0.0;
    if (jacobi)
        memcpy(pold, p, bytes);                 // Jacobi: compute from the old values only
    double *q = jacobi ? pold : p;              // Gauss-Seidel/SOR: always the newest values
#define Q(i, j) q[(j) * (imax + 2) + (i)]
    for (int pass = 0; pass < (redblack ? 2 : 1); ++pass) {
        for (int j = 1; j <= jmax; ++j) {
            // red-black: in pass 0 the points with i + j even, in pass 1 those with i + j odd
            int istart = redblack ? 1 + (j + 1 + pass) % 2 : 1, istep = redblack ? 2 : 1;
            for (int i = istart; i <= imax; i += istep) {
                double r = RHS(i, j) - ((Q(i - 1, j) - 2.0 * Q(i, j) + Q(i + 1, j)) * idx2 +
                                        (Q(i, j - 1) - 2.0 * Q(i, j) + Q(i, j + 1)) * idy2);
                P(i, j) = Q(i, j) - factor * r;
                res += r * r;
            }
        }
    }
    // homogeneous Neumann boundary conditions in the ghost cells
    for (int i = 1; i <= imax; ++i) {
        P(i, 0)        = P(i, 1);
        P(i, jmax + 1) = P(i, jmax);
    }
    for (int j = 1; j <= jmax; ++j) {
        P(0, j)        = P(1, j);
        P(imax + 1, j) = P(imax, j);
    }
    res /= (double)imax * jmax;                 // mean square residual
    ++it;
}

// the solution is unique only up to a constant: compare after removing the mean
double mean = 0.0, err = 0.0;
for (int j = 1; j <= jmax; ++j)
    for (int i = 1; i <= imax; ++i)
        mean += P(i, j);
mean /= (double)imax * jmax;
for (int j = 1; j <= jmax; ++j)
    for (int i = 1; i <= imax; ++i) {
        double e = fabs(P(i, j) - mean - cos(PI * (i - 0.5) * dx) * cos(PI * (j - 0.5) * dy));
        if (e > err) err = e;
    }
printf("%-6s omega %.3f n %4d: %6d iterations, max. error %.3e\n", method, w, n, it, err);
free(p); free(pold); free(rhs);

> 🤔 **Predict:** How do Jacobi, Gauss–Seidel and SOR compare? And how does the error change when the grid is refined from 32 to 64 to 128 cells per direction?

The following cell runs the program for many values of $\omega$ — this takes a few seconds.

In [ ]:
# iterations vs. omega (runs the program of the previous cell many times)
import re

src = open('code/cfd-poisson.c').read()

def run(method, omega, n=64):
    get_ipython().run_cell_magic('c', f'-r -f lm -a {method} {omega} {n} -O run_output', src)
    return int(re.search(r'(\d+) iterations', run_output).group(1))

omegas = [1.0, 1.2, 1.4, 1.6, 1.7, 1.8, 1.85, 1.9, 1.93, 1.95, 1.96, 1.97, 1.98, 1.99]
sor = [run('sor', w) for w in omegas]
rb = [run('rb', w) for w in omegas]
jacobi = run('jacobi', 1.0)

print('grid refinement with SOR:')
for n in (32, 64, 128):
    get_ipython().run_cell_magic('c', f'-r -f lm -a sor 1.9 {n} -O run_output', src)

In [ ]:
# plot the iterations vs. omega (requires matplotlib)
surface, ink, muted, grid, axis = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df', '#a3a29c'
fig, ax = plt.subplots(figsize=(8.5, 4.5), facecolor=surface)
ax.set_facecolor(surface)
# legend: minimum number of iterations and the omega where it is reached
best = lambda its: f'min. {min(its)} iterations at ω = {omegas[its.index(min(its))]}'
ax.plot(omegas, sor, color='#2a78d6', linewidth=2, marker='o', markersize=5, label=f'SOR (lexicographic): {best(sor)}')
ax.plot(omegas, rb, color='#eb6834', linewidth=2, marker='o', markersize=5, label=f'red-black SOR: {best(rb)}')
ax.axhline(jacobi, color=muted, linewidth=1.2, linestyle='--')
ax.annotate(f'Jacobi: {jacobi} iterations', (1.3, jacobi), xytext=(0, 5), textcoords='offset points', color=ink, fontsize=10)
w_est = 2 / (1 + np.sin(np.pi / 64))
ax.axvline(w_est, color=muted, linewidth=1, linestyle=':')
ax.annotate(f'2 / (1 + sin(πh)) = {w_est:.3f}', (w_est, max(sor) * 0.55), xytext=(-8, 0), textcoords='offset points',
            ha='right', color=ink, fontsize=10)
ax.set_yscale('log')
ax.set_xlabel('ω', color=muted)
ax.set_title('Iterations to reach ‖r‖ < 10⁻⁶ — 64 × 64 grid', loc='left', fontsize=12, color=ink)
ax.grid(True, which='major', color=grid, linewidth=0.8)
ax.tick_params(colors=muted, which='both')
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
for side in ('left', 'bottom'):
    ax.spines[side].set_color(axis)
ax.legend(frameon=False, labelcolor=ink)
fig.tight_layout()
plt.show()

* Gauss–Seidel ($\omega = 1$) needs somewhat fewer iterations than Jacobi; SOR with a good $\omega$ is an order of magnitude faster.
* The optimum is **sharp**, close to 2, and the iterations grow quickly beyond it.
  For the Dirichlet problem, theory gives $\omega_{opt} = 2/(1 + \sin \pi h)$; for our pure Neumann problem, the measured optimum lies higher.
  In practice, $\omega$ is found by experiment.
* The error decreases by a factor of 4 when the grid is refined by a factor of 2: the discretization is second-order accurate.
  It is the **discretization** error — iterating further doesn't reduce it.

The drawbacks of SOR: the optimal $\omega$ is hard to determine, the sequential dependency (each point needs its freshly updated neighbors) runs poorly on modern processors — it prevents vectorization — and it is difficult to parallelize.

### Red-Black Ordering

Color the points like a checkerboard: red if $i + j$ is even, black otherwise.

![Red-black ordering of an 8 by 8 grid: in pass 1 all red points are updated, in pass 2 all black points. The five-point stencil of a red point only touches black points and vice versa](img/cfd-redblack.svg)

A red point's stencil only touches black neighbors and vice versa.
So one iteration updates **all red points** (from the black values), then **all black points** (from the new red values) — within each pass, all updates are independent: they can be vectorized and parallelized without changing the result.
As the plot shows, red-black SOR even converges faster here than lexicographic SOR.

## MPI Parallelization of the Poisson Solver

### 1D Domain Decomposition

The simplest decomposition for a 2D grid stored row by row: every process gets a block of **complete rows**, plus one **ghost row** below and above, which holds a copy of the neighbor's boundary row.
In C, rows are contiguous in memory, so a ghost row is a plain array of `imax + 2` doubles — no derived data types needed.

![Three ranks, each owning a block of rows with a ghost row below and above. Exchange: every rank sends its first own row down and its last own row up. Shift: data travels only upwards](img/cfd-halo.svg)

Before each sweep, the processes **exchange** their boundary rows with both neighbors.
With `MPI_Sendrecv()` and `MPI_PROC_NULL` as the neighbor at the domain boundary, this takes two calls without any special cases:

```c
MPI_Sendrecv(&P(0, jmaxLocal), imax + 2, MPI_DOUBLE, top,    0,     // last own row up ...
             &P(0, 0),         imax + 2, MPI_DOUBLE, bottom, 0, …); // ... bottom ghost row from below
MPI_Sendrecv(&P(0, 1),             imax + 2, MPI_DOUBLE, bottom, 1, // first own row down ...
             &P(0, jmaxLocal + 1), imax + 2, MPI_DOUBLE, top,    1, …); // ... top ghost row from above
```

Further changes compared with the sequential code:

* the processes at the bottom and top set the boundary conditions there, all processes on the left and right,
* the residual is a **global sum**: `MPI_Allreduce()` — and must be **reset in every iteration**,
* the coordinates of a local row follow from the number of rows below, `jstart`, which is a prefix sum of the local sizes — `rank * jmaxLocal` is wrong when the rows don't divide evenly.

### Does the Parallelization Change the Algorithm?

Every parallelization should be checked for one question: *does the distributed processing change the algorithm?*
Lexicographic SOR with one exchange per sweep does: in the sequential sweep, the first row of a process uses the **new** values of the row below — in the parallel one, the ghost row holds the values of the previous iteration.
The method becomes a mix of Gauss–Seidel within and Jacobi between the processes.
Red-black SOR with an exchange **before each color** computes exactly what the sequential version computes.

The following program implements both (`-a lex` or `-a rb`, then ω and n).

> 🤔 **Predict:** How do the iteration counts change with the number of processes for both variants?

In [ ]:
%%mpi -n 4 -i string.h math.h -f lm -a rb 1.95 64 -o code/cfd-poisson-mpi.c
// arguments: method (lex | rb), omega, n (cells per direction)
const char *method = argc > 1 ? argv[1] : "rb";
const double omega = argc > 2 ? atof(argv[2]) : 1.95;
const int n        = argc > 3 ? atoi(argv[3]) : 64;

const int imax = n, jmax = n;
const double dx = 1.0 / imax, dy = 1.0 / jmax, eps = 1e-6;
const int itermax = 100000;
const double PI = 3.14159265358979323846;

// 1D decomposition in j: every rank owns jmaxLocal complete rows, after the jstart rows of the ranks below
const int jmaxLocal = jmax / size + (rank < jmax % size ? 1 : 0);
const int jstart    = rank * (jmax / size) + (rank < jmax % size ? rank : jmax % size);
const int bottom    = rank > 0        ? rank - 1 : MPI_PROC_NULL;
const int top       = rank < size - 1 ? rank + 1 : MPI_PROC_NULL;

const size_t bytes = (size_t)(imax + 2) * (jmaxLocal + 2) * sizeof(double);
double *p = calloc(1, bytes), *rhs = calloc(1, bytes);
#define P(i, j)   p[(j) * (imax + 2) + (i)]
#define RHS(i, j) rhs[(j) * (imax + 2) + (i)]

// local row j is the global row jstart + j
for (int j = 1; j <= jmaxLocal; ++j)
    for (int i = 1; i <= imax; ++i)
        RHS(i, j) = -2.0 * PI * PI * cos(PI * (i - 0.5) * dx) * cos(PI * (jstart + j - 0.5) * dy);

// halo exchange of the ghost rows: rows are contiguous, MPI_PROC_NULL handles the global boundary
#define EXCHANGE()                                                                                  \
    do {                                                                                            \
        MPI_Sendrecv(&P(0, jmaxLocal), imax + 2, MPI_DOUBLE, top, 0,                                \
                     &P(0, 0), imax + 2, MPI_DOUBLE, bottom, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);   \
        MPI_Sendrecv(&P(0, 1), imax + 2, MPI_DOUBLE, bottom, 1,                                     \
                     &P(0, jmaxLocal + 1), imax + 2, MPI_DOUBLE, top, 1, MPI_COMM_WORLD,              \
                     MPI_STATUS_IGNORE);                                                            \
    } while (0)

const int redblack = strcmp(method, "rb") == 0;
const double dx2 = dx * dx, dy2 = dy * dy, idx2 = 1.0 / dx2, idy2 = 1.0 / dy2;
const double factor = omega * 0.5 * dx2 * dy2 / (dx2 + dy2);

int it = 0;
double res = 1.0;
while (res >= eps * eps && it < itermax) {
    res = 0.0;                                  // reset in every iteration!
    if (!redblack)
        EXCHANGE();                             // lexicographic: once per sweep
    for (int pass = 0; pass < (redblack ? 2 : 1); ++pass) {
        if (redblack)
            EXCHANGE();                         // red-black: before each color
        for (int j = 1; j <= jmaxLocal; ++j) {
            // the color depends on the global row index
            int istart = redblack ? 1 + (jstart + j + 1 + pass) % 2 : 1, istep = redblack ? 2 : 1;
            for (int i = istart; i <= imax; i += istep) {
                double r = RHS(i, j) - ((P(i - 1, j) - 2.0 * P(i, j) + P(i + 1, j)) * idx2 +
                                        (P(i, j - 1) - 2.0 * P(i, j) + P(i, j + 1)) * idy2);
                P(i, j) -= factor * r;
                res += r * r;
            }
        }
    }
    // Neumann conditions only at the global boundary
    if (bottom == MPI_PROC_NULL)
        for (int i = 1; i <= imax; ++i) P(i, 0) = P(i, 1);
    if (top == MPI_PROC_NULL)
        for (int i = 1; i <= imax; ++i) P(i, jmaxLocal + 1) = P(i, jmaxLocal);
    for (int j = 1; j <= jmaxLocal; ++j) {
        P(0, j)        = P(1, j);
        P(imax + 1, j) = P(imax, j);
    }
    MPI_Allreduce(MPI_IN_PLACE, &res, 1, MPI_DOUBLE, MPI_SUM, MPI_COMM_WORLD);
    res /= (double)imax * jmax;
    ++it;
    if (!isfinite(res)) {                       // the iteration diverged
        if (rank == 0)
            printf("%d processes, %-3s omega %.2f n %d: diverged after %d iterations\n", size, method, omega, n, it);
        break;
    }
}
if (!isfinite(res)) {
    free(p);
    free(rhs);
    MPI_Finalize();
    return 0;
}

// error against the exact solution, after removing the mean
double mean = 0.0, err = 0.0;
for (int j = 1; j <= jmaxLocal; ++j)
    for (int i = 1; i <= imax; ++i)
        mean += P(i, j);
MPI_Allreduce(MPI_IN_PLACE, &mean, 1, MPI_DOUBLE, MPI_SUM, MPI_COMM_WORLD);
mean /= (double)imax * jmax;
for (int j = 1; j <= jmaxLocal; ++j)
    for (int i = 1; i <= imax; ++i) {
        double e = fabs(P(i, j) - mean - cos(PI * (i - 0.5) * dx) * cos(PI * (jstart + j - 0.5) * dy));
        if (e > err) err = e;
    }
MPI_Allreduce(MPI_IN_PLACE, &err, 1, MPI_DOUBLE, MPI_MAX, MPI_COMM_WORLD);
if (rank == 0)
    printf("%d processes, %-3s omega %.2f n %d: %5d iterations, max. error %.3e\n", size, method, omega, n, it, err);
free(p);
free(rhs);

In [ ]:
# run both variants with 1 to 4 processes
src = open('code/cfd-poisson-mpi.c').read()
for method, omega in (('rb', 1.95), ('lex', 1.95), ('lex', 1.7)):
    for n in (1, 2, 3, 4):
        get_ipython().run_cell_magic('mpi', f'-r -n {n} -f lm -a {method} {omega} 64', src)

Red-black SOR needs the same number of iterations for any number of processes, and gives the same error.
Lexicographic SOR needs more iterations the more processes there are — and with $\omega = 1.95$, which is close to the sequential optimum, it **diverges** for 3 and more processes: the stale ghost values destroy the over-relaxation.
A parallel program whose convergence depends on the number of processes is hard to debug and to benchmark; red-black SOR avoids the problem.

## Exercises

### Exercise 1: A 2D Decomposition of the Poisson Solver

Change the parallel Poisson solver to a **2D** domain decomposition:

* create a 2D Cartesian process grid (`MPI_Dims_create()`, `MPI_Cart_create()`, `MPI_Cart_shift()`),
* compute the size and the offset of the local block in both dimensions,
* exchange rows and **columns** — a column is a vector data type with a stride of the row length,
* set the boundary conditions only at the domain boundary, and compute the colors from the global indices.

Check that the iterations and the error are the same as before for any number of processes (try `-n 6` and `-n 9`).

<details>
<summary>Solution</summary>

```c
const double omega = argc > 1 ? atof(argv[1]) : 1.95;
const int n        = argc > 2 ? atoi(argv[2]) : 64;
const int imax = n, jmax = n;
const double dx = 1.0 / imax, dy = 1.0 / jmax, eps = 1e-6;
const int itermax = 100000;
const double PI = 3.14159265358979323846;

// 2D process grid: dimension 0 along j, dimension 1 along i
int dims[2] = { 0, 0 }, periods[2] = { 0, 0 }, coords[2];
MPI_Dims_create(size, 2, dims);
MPI_Comm comm;
MPI_Cart_create(MPI_COMM_WORLD, 2, dims, periods, 0, &comm);
MPI_Comm_rank(comm, &rank);
MPI_Cart_coords(comm, rank, 2, coords);
int bottom, top, left, right;
MPI_Cart_shift(comm, 0, 1, &bottom, &top);
MPI_Cart_shift(comm, 1, 1, &left, &right);

#define SIZE_OF(c, N, P)  ((N) / (P) + ((c) < (N) % (P) ? 1 : 0))
#define START_OF(c, N, P) ((c) * ((N) / (P)) + ((c) < (N) % (P) ? (c) : (N) % (P)))
const int jmaxLocal = SIZE_OF(coords[0], jmax, dims[0]), jstart = START_OF(coords[0], jmax, dims[0]);
const int imaxLocal = SIZE_OF(coords[1], imax, dims[1]), istart = START_OF(coords[1], imax, dims[1]);
const int ld = imaxLocal + 2;

double *p = calloc((size_t)ld * (jmaxLocal + 2), sizeof(double));
double *rhs = calloc((size_t)ld * (jmaxLocal + 2), sizeof(double));
#define P(i, j)   p[(j) * ld + (i)]
#define RHS(i, j) rhs[(j) * ld + (i)]
for (int j = 1; j <= jmaxLocal; ++j)
    for (int i = 1; i <= imaxLocal; ++i)
        RHS(i, j) = -2.0 * PI * PI * cos(PI * (istart + i - 0.5) * dx) * cos(PI * (jstart + j - 0.5) * dy);

// a row is contiguous, a column is a vector
MPI_Datatype row, column;
MPI_Type_contiguous(imaxLocal, MPI_DOUBLE, &row);
MPI_Type_vector(jmaxLocal, 1, ld, MPI_DOUBLE, &column);
MPI_Type_commit(&row);
MPI_Type_commit(&column);

#define EXCHANGE()                                                                                          \
    do {                                                                                                    \
        MPI_Sendrecv(&P(1, jmaxLocal), 1, row, top, 0, &P(1, 0), 1, row, bottom, 0, comm, MPI_STATUS_IGNORE);         \
        MPI_Sendrecv(&P(1, 1), 1, row, bottom, 1, &P(1, jmaxLocal + 1), 1, row, top, 1, comm, MPI_STATUS_IGNORE);     \
        MPI_Sendrecv(&P(imaxLocal, 1), 1, column, right, 2, &P(0, 1), 1, column, left, 2, comm, MPI_STATUS_IGNORE);   \
        MPI_Sendrecv(&P(1, 1), 1, column, left, 3, &P(imaxLocal + 1, 1), 1, column, right, 3, comm, MPI_STATUS_IGNORE); \
    } while (0)

const double dx2 = dx * dx, dy2 = dy * dy, idx2 = 1.0 / dx2, idy2 = 1.0 / dy2;
const double factor = omega * 0.5 * dx2 * dy2 / (dx2 + dy2);
int it = 0;
double res = 1.0;
while (res >= eps * eps && it < itermax) {
    res = 0.0;
    for (int pass = 0; pass < 2; ++pass) {
        EXCHANGE();
        for (int j = 1; j <= jmaxLocal; ++j)
            // the color depends on the global indices istart + i and jstart + j
            for (int i = 1 + (istart + jstart + j + 1 + pass) % 2; i <= imaxLocal; i += 2) {
                double r = RHS(i, j) - ((P(i - 1, j) - 2.0 * P(i, j) + P(i + 1, j)) * idx2 +
                                        (P(i, j - 1) - 2.0 * P(i, j) + P(i, j + 1)) * idy2);
                P(i, j) -= factor * r;
                res += r * r;
            }
    }
    if (bottom == MPI_PROC_NULL) for (int i = 1; i <= imaxLocal; ++i) P(i, 0) = P(i, 1);
    if (top    == MPI_PROC_NULL) for (int i = 1; i <= imaxLocal; ++i) P(i, jmaxLocal + 1) = P(i, jmaxLocal);
    if (left   == MPI_PROC_NULL) for (int j = 1; j <= jmaxLocal; ++j) P(0, j) = P(1, j);
    if (right  == MPI_PROC_NULL) for (int j = 1; j <= jmaxLocal; ++j) P(imaxLocal + 1, j) = P(imaxLocal, j);
    MPI_Allreduce(MPI_IN_PLACE, &res, 1, MPI_DOUBLE, MPI_SUM, comm);
    res /= (double)imax * jmax;
    ++it;
}

double mean = 0.0, err = 0.0;
for (int j = 1; j <= jmaxLocal; ++j)
    for (int i = 1; i <= imaxLocal; ++i)
        mean += P(i, j);
MPI_Allreduce(MPI_IN_PLACE, &mean, 1, MPI_DOUBLE, MPI_SUM, comm);
mean /= (double)imax * jmax;
for (int j = 1; j <= jmaxLocal; ++j)
    for (int i = 1; i <= imaxLocal; ++i) {
        double e = fabs(P(i, j) - mean - cos(PI * (istart + i - 0.5) * dx) * cos(PI * (jstart + j - 0.5) * dy));
        if (e > err) err = e;
    }
MPI_Allreduce(MPI_IN_PLACE, &err, 1, MPI_DOUBLE, MPI_MAX, comm);
if (rank == 0)
    printf("%d x %d processes: %d iterations, max. error %.3e\n", dims[0], dims[1], it, err);

MPI_Type_free(&row);
MPI_Type_free(&column);
MPI_Comm_free(&comm);
free(p);
free(rhs);
```
</details>

In [ ]:
%%mpi -n 4 -i string.h math.h -f lm -a rb 1.95 64
// TODO: change the 1D decomposition into a 2D decomposition
// arguments: method (lex | rb), omega, n (cells per direction)
const char *method = argc > 1 ? argv[1] : "rb";
const double omega = argc > 2 ? atof(argv[2]) : 1.95;
const int n        = argc > 3 ? atoi(argv[3]) : 64;

const int imax = n, jmax = n;
const double dx = 1.0 / imax, dy = 1.0 / jmax, eps = 1e-6;
const int itermax = 100000;
const double PI = 3.14159265358979323846;

// 1D decomposition in j: every rank owns jmaxLocal complete rows, after the jstart rows of the ranks below
const int jmaxLocal = jmax / size + (rank < jmax % size ? 1 : 0);
const int jstart    = rank * (jmax / size) + (rank < jmax % size ? rank : jmax % size);
const int bottom    = rank > 0        ? rank - 1 : MPI_PROC_NULL;
const int top       = rank < size - 1 ? rank + 1 : MPI_PROC_NULL;

const size_t bytes = (size_t)(imax + 2) * (jmaxLocal + 2) * sizeof(double);
double *p = calloc(1, bytes), *rhs = calloc(1, bytes);
#define P(i, j)   p[(j) * (imax + 2) + (i)]
#define RHS(i, j) rhs[(j) * (imax + 2) + (i)]

// local row j is the global row jstart + j
for (int j = 1; j <= jmaxLocal; ++j)
    for (int i = 1; i <= imax; ++i)
        RHS(i, j) = -2.0 * PI * PI * cos(PI * (i - 0.5) * dx) * cos(PI * (jstart + j - 0.5) * dy);

// halo exchange of the ghost rows: rows are contiguous, MPI_PROC_NULL handles the global boundary
#define EXCHANGE()                                                                                  \
    do {                                                                                            \
        MPI_Sendrecv(&P(0, jmaxLocal), imax + 2, MPI_DOUBLE, top, 0,                                \
                     &P(0, 0), imax + 2, MPI_DOUBLE, bottom, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);   \
        MPI_Sendrecv(&P(0, 1), imax + 2, MPI_DOUBLE, bottom, 1,                                     \
                     &P(0, jmaxLocal + 1), imax + 2, MPI_DOUBLE, top, 1, MPI_COMM_WORLD,              \
                     MPI_STATUS_IGNORE);                                                            \
    } while (0)

const int redblack = strcmp(method, "rb") == 0;
const double dx2 = dx * dx, dy2 = dy * dy, idx2 = 1.0 / dx2, idy2 = 1.0 / dy2;
const double factor = omega * 0.5 * dx2 * dy2 / (dx2 + dy2);

int it = 0;
double res = 1.0;
while (res >= eps * eps && it < itermax) {
    res = 0.0;                                  // reset in every iteration!
    if (!redblack)
        EXCHANGE();                             // lexicographic: once per sweep
    for (int pass = 0; pass < (redblack ? 2 : 1); ++pass) {
        if (redblack)
            EXCHANGE();                         // red-black: before each color
        for (int j = 1; j <= jmaxLocal; ++j) {
            // the color depends on the global row index
            int istart = redblack ? 1 + (jstart + j + 1 + pass) % 2 : 1, istep = redblack ? 2 : 1;
            for (int i = istart; i <= imax; i += istep) {
                double r = RHS(i, j) - ((P(i - 1, j) - 2.0 * P(i, j) + P(i + 1, j)) * idx2 +
                                        (P(i, j - 1) - 2.0 * P(i, j) + P(i, j + 1)) * idy2);
                P(i, j) -= factor * r;
                res += r * r;
            }
        }
    }
    // Neumann conditions only at the global boundary
    if (bottom == MPI_PROC_NULL)
        for (int i = 1; i <= imax; ++i) P(i, 0) = P(i, 1);
    if (top == MPI_PROC_NULL)
        for (int i = 1; i <= imax; ++i) P(i, jmaxLocal + 1) = P(i, jmaxLocal);
    for (int j = 1; j <= jmaxLocal; ++j) {
        P(0, j)        = P(1, j);
        P(imax + 1, j) = P(imax, j);
    }
    MPI_Allreduce(MPI_IN_PLACE, &res, 1, MPI_DOUBLE, MPI_SUM, MPI_COMM_WORLD);
    res /= (double)imax * jmax;
    ++it;
    if (!isfinite(res)) {                       // the iteration diverged
        if (rank == 0)
            printf("%d processes, %-3s omega %.2f n %d: diverged after %d iterations\n", size, method, omega, n, it);
        break;
    }
}
if (!isfinite(res)) {
    free(p);
    free(rhs);
    MPI_Finalize();
    return 0;
}

// error against the exact solution, after removing the mean
double mean = 0.0, err = 0.0;
for (int j = 1; j <= jmaxLocal; ++j)
    for (int i = 1; i <= imax; ++i)
        mean += P(i, j);
MPI_Allreduce(MPI_IN_PLACE, &mean, 1, MPI_DOUBLE, MPI_SUM, MPI_COMM_WORLD);
mean /= (double)imax * jmax;
for (int j = 1; j <= jmaxLocal; ++j)
    for (int i = 1; i <= imax; ++i) {
        double e = fabs(P(i, j) - mean - cos(PI * (i - 0.5) * dx) * cos(PI * (jstart + j - 0.5) * dy));
        if (e > err) err = e;
    }
MPI_Allreduce(MPI_IN_PLACE, &err, 1, MPI_DOUBLE, MPI_MAX, MPI_COMM_WORLD);
if (rank == 0)
    printf("%d processes, %-3s omega %.2f n %d: %5d iterations, max. error %.3e\n", size, method, omega, n, it, err);
free(p);
free(rhs);

## Summary

* The Poisson equation is discretized with the five-point stencil on a cell-centered grid; ghost cells hold the boundary conditions. With pure Neumann conditions, the solution is unique only up to a constant.
* Jacobi, Gauss–Seidel and SOR solve the linear system by relaxation; SOR with a good ω is much faster, red-black ordering makes it parallel.
* A 1D domain decomposition in rows needs ghost rows, a halo exchange (`MPI_Sendrecv()` with `MPI_PROC_NULL`) and global reductions (`MPI_Allreduce()`).
* The parallelization must not change the algorithm: lexicographic SOR does, red-black SOR with an exchange per color doesn't.